# Field - Python

All 12 Python examples from [docs/field.md](https://platob.github.io/yggdryl/field/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

In [ ]:
from yggdryl import DataType, Field

field = Field("price", "decimal(18, 6)", nullable=False)

assert field.name == "price"
assert field.data_type == DataType("decimal(18, 6)")
assert field.nullable is False
assert len(field) == 0

assert Field.from_str(str(field)) == field
assert Field.from_str("price decimal(18, 6) NOT NULL") == field

## A non-null struct field is the schema

In [ ]:
from yggdryl import DataType, Field, fields

schema = Field(
    "trade",
    DataType.from_fields([
        fields.int64("id", nullable=False),
        fields.utf8("symbol"),
    ]),
    nullable=False,
)

children = schema.data_type
assert len(children) == 2
assert "symbol" in children
assert children["id"].nullable is False
assert children[1].name == "symbol"
assert [child.name for child in children] == ["id", "symbol"]

### Item access reaches a child, never metadata

In [ ]:
from yggdryl import DataType, Field

order = Field(
    "order",
    DataType.from_fields([
        Field("id", "int64", nullable=False),
        Field(
            "line",
            DataType.from_fields([Field("price", "float64", nullable=False)]),
            nullable=False,
        ),
    ]),
    nullable=False,
    metadata={"owner": "trading"},
)

# A child by name, by position, negatively, and two levels down.
assert order["id"].data_type == DataType("int64")
assert order[-1].name == "line"
assert order["line"]["price"].data_type == DataType("float64")

# The DataType answers the same way, and children drive len/iter/in.
assert order.data_type["id"].name == "id"
assert len(order) == 2
assert [child.name for child in order] == ["id", "line"]
assert "line" in order

# An unknown name appends; a position replaces only.
order["venue"] = Field("venue", "utf8")
assert len(order) == 3
order[0] = Field("id", "utf8", nullable=False)
assert order["id"].data_type == DataType("utf8")
del order["venue"]
assert len(order) == 2

# Metadata is reached through its view, never by subscripting the node.
assert order.metadata["owner"] == "trading"
try:
    order["owner"]
except KeyError:
    pass

## Metadata is a mapping

In [ ]:
from yggdryl import Field

field = Field("price", "float64", nullable=False, metadata={"venue": "XPAR"})
# Metadata lives on `field.metadata`, a live mapping view. Subscripting the
# field itself reaches a nested *child*, not a metadata key.
field.metadata["currency"] = "EUR"
field.metadata.update(source="exchange")

assert len(field.metadata) == 3
assert "venue" in field.metadata
assert field.metadata["venue"] == "XPAR"
assert field.metadata.get("missing") is None
assert list(field.metadata.items()) == [
    ("currency", "EUR"),
    ("source", "exchange"),
    ("venue", "XPAR"),
]

del field.metadata["venue"]
assert list(field.metadata.keys()) == ["currency", "source"]

## Reserved keys and protocol properties

In [ ]:
from yggdryl import Field, MimeType

field = Field("payload", "binary", nullable=False)

field.set_parquet_field_id(17)
field.metadata["field:init"] = "false"
field.set_content_type("application/json; charset=utf-8")
field.set_property("postgres", "type", "jsonb")

assert field.parquet_field_id == 17
assert field.metadata["PARQUET:field_id"] == "17"
assert field.metadata["field:init"] == "false"

assert field.mime_type == MimeType.JSON
assert field.get_property("https", "Content-Type") == field.content_type
assert field.metadata["http:content-type"] == field.content_type
assert list(field.property_iter("postgres")) == [("type", "jsonb")]

## One protocol at a time

In [ ]:
from yggdryl import Field

field = Field("price", "int64", nullable=False)

field.iceberg["doc"] = "closing price"
field.iceberg.update({"schema-id": "3", "field-id": "7"})
field.postgres["type"] = "numeric"

assert field.iceberg["doc"] == "closing price"
assert field.iceberg.key("doc") == "iceberg:doc"
assert len(field.iceberg) == 3
assert not field.mysql

# It is a view of the one metadata mapping, not a copy of part of it.
assert field.metadata["iceberg:doc"] == "closing price"
assert len(field.metadata) == 4
assert dict(field.iceberg.items())["field-id"] == "7"

del field.iceberg["field-id"]
assert "field-id" not in field.iceberg
assert field.protocol("postgres")["type"] == "numeric"

## A field can be a partition column

In [ ]:
from yggdryl import DataType, Field

schema = Field(
    "row",
    DataType.from_fields([
        Field("year", "int32", nullable=False),
        Field("venue", "string", nullable=False),
        Field("price", "int64", nullable=False),
    ]),
    nullable=False,
).with_partition_fields(["year", "venue"])

assert schema.has_partition_fields
assert schema.partition_field_names == ["year", "venue"]
assert schema.data_type["year"].is_partition
assert not schema.data_type["price"].is_partition

assert len(schema.without_partition_fields().data_type) == 1
assert len(schema.only_partition_fields().data_type) == 2

## Typed field aliases

In [ ]:
from yggdryl import Field, fields

id_field = fields.int64("id", nullable=False)
symbol = fields.utf8("symbol", metadata={"source": "feed"})
at = fields.timestamp("at", "us", nullable=False)

assert isinstance(id_field, Field)
assert str(id_field.data_type) == "int64"
assert symbol.metadata["source"] == "feed"
assert str(at.data_type) == "timestamp(us)"

## Comparing two fields

In [ ]:
from yggdryl import Field

left = Field("price", "float64", nullable=False, metadata={"venue": "XPAR"})
right = Field("price", "float64", metadata={"venue": "XNAS"})

assert not left.equals(right)
assert list(left.show_diffs(right)) == [
    "≠ $.nullable: false → true",
    '≠ $.metadata["venue"]: "XPAR" → "XNAS"',
]
assert left.show_diff(left) == "✓ equal"
assert left.show_diff(left, return_equal=False) == ""

## Casting Arrow data through a field

In [ ]:
import pyarrow as pa
from yggdryl import Field

field = Field("id", "int64", nullable=False)

ids = field.cast_arrow_array(pa.array(["1", "2"]))
assert ids.equals(pa.array([1, 2], type=pa.int64()))

# safe nulls a failed conversion; a non-null field then defaults it.
repaired = field.cast_arrow_array(pa.array(["1", "not a number"]))
assert repaired.equals(pa.array([1, 0], type=pa.int64()))
assert repaired.null_count == 0

try:
    field.cast_arrow_array(pa.array(["1", "not a number"]), safe=False)
except ValueError:
    pass
else:
    raise AssertionError("an unsafe cast must fail")

In [ ]:
import pyarrow as pa
from yggdryl import DataType, Field, fields

schema = Field(
    "trade",
    DataType.from_fields([
        fields.int64("id", nullable=False),
        fields.utf8("symbol"),
    ]),
    nullable=False,
)

source = pa.record_batch({
    "symbol": pa.array(["ACME"]),
    "id": pa.array([7], type=pa.int32()),
})

batch = schema.cast_arrow_batch(source)
assert batch.schema.names == ["id", "symbol"]
assert batch.column("id").type == pa.int64()

## The generic cast

In [ ]:
import pyarrow as pa

from yggdryl import DataType, Field

schema = Field("row", DataType("struct<id: int64, symbol: string>"), False)
table = pa.table({"id": pa.array([1, 2], pa.int32()), "symbol": ["AAPL", "MSFT"]})

# A table comes back a table, a reader a reader, a frame a frame.
cast = schema.cast_arrow(table)
assert cast.schema.field("id").type == pa.int64()

# The generic name also takes plain values, as the typed scalar.
price = Field("price", DataType("int64"), False)
assert price.cast(5).as_py() == 5